# Experiment 3 :: Stemming, Lemmatization and Regular Expressions

- 3.1 Stemming with the Porter stemmer
- 3.2 Lemmatization with WordNet, with and without POS tags
- 3.3 Regular expressions: extract emails, URLs, mobile numbers, hashtags and mentions

In [1]:
!pip install -q nltk

zsh:1: command not found: pip


In [2]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package wordnet to /Users/apple/nltk_data...


[nltk_data] Downloading package omw-1.4 to /Users/apple/nltk_data...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/apple/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/apple/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

## Input data

In [3]:
stemming_words = open('3.1_stemming_data.txt', encoding='utf-8').read().split()
lemmatization_words = open('3.2_lemmatization_data.txt', encoding='utf-8').read().split()
regex_text = open('3.3_regex_data.txt', encoding='utf-8').read().strip()

print('Words for stemming:', stemming_words)
print()
print('Words for lemmatization:', lemmatization_words)
print()
print('Text for regex extraction:')
print(regex_text)

Words for stemming: ['playing', 'played', 'plays', 'studies', 'studying', 'connected', 'connection', 'computers']

Words for lemmatization: ['cats', 'dogs', 'running', 'runs', 'ran', 'studies', 'studying', 'better', 'children', 'mice', 'went', 'ate', 'leaves', 'caring']

Text for regex extraction:
Welcome to the Natural Language Processing workshop! For more information, visit https://www.nlpworkshop.com or www.python.org. You can contact the coordinator at nlpworkshop@gmail.com or support@python.org. For registration, call +91-9876543210 or 9123456789. Follow us on social media @NLPWorkshop and @PythonLearner. Share your experience using #NLP, #Python, and #MachineLearning. You can also visit https://github.com/NLPWorkshop for the latest updates.


## 3.1 Stemming using the Porter stemmer

In [4]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

print(f"{'WORD':<15}{'STEM'}")
for word in stemming_words:
    print(f'{word:<15}{stemmer.stem(word)}')

WORD           STEM
playing        play
played         play
plays          play
studies        studi
studying       studi
connected      connect
connection     connect
computers      comput


The stemmer only strips suffixes, it never checks a dictionary. That is why `connected` and
`connection` correctly collapse to `connect`, while `studies` and `studying` both end up at
`studi`, which is not an English word at all.

## 3.2 Lemmatization using WordNet

`WordNetLemmatizer` assumes every word is a noun unless told otherwise, so `running` stays
`running`. Tagging each word first and passing the part of speech gives the lemmatizer the
context it needs. The POS column shows WordNet's tag: n = noun, v = verb, a = adjective, r = adverb.

In [5]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

POS_MAP = {'J': wordnet.ADJ, 'N': wordnet.NOUN, 'V': wordnet.VERB, 'R': wordnet.ADV}

def wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1]
    return POS_MAP.get(tag[0], wordnet.NOUN)

print(f"{'WORD':<15}{'POS':<6}{'AS NOUN (DEFAULT)':<20}{'WITH POS TAG'}")
for word in lemmatization_words:
    pos = wordnet_pos(word)
    print(f'{word:<15}{pos:<6}{lemmatizer.lemmatize(word):<20}{lemmatizer.lemmatize(word, pos)}')

WORD           POS   AS NOUN (DEFAULT)   WITH POS TAG
cats           n     cat                 cat
dogs           n     dog                 dog
running        v     running             run
runs           n     run                 run
ran            n     ran                 ran
studies        n     study               study
studying       v     studying            study
better         r     better              well
children       n     child               child
mice           n     mouse               mouse
went           v     went                go
ate            n     ate                 ate
leaves         n     leaf                leaf
caring         v     caring              care


Where stemming produced `studi`, the lemmatizer returns the dictionary form `study`, and
irregular forms resolve to their dictionary entries: `went` becomes `go`, `mice` becomes
`mouse`, `children` becomes `child`, and the comparative `better` becomes `well`. The table
also shows the technique's weak spot: each word is tagged in isolation, so `ran` and `ate`
are mistaken for nouns and pass through unchanged. Lemmatization is only as good as the POS
information it is given.

## 3.3 Extraction with regular expressions

In [6]:
import re

PATTERNS = {
    'Emails': r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
    'URLs': r'https?://\S+|www\.\S+',
    'Mobile numbers': r'\+?\d{1,3}[-.\s]?\d{10}|\b\d{10}\b',
    'Hashtags': r'#\w+',
    'Mentions': r'(?<![\w.%+-])@\w+',
}

for name, pattern in PATTERNS.items():
    matches = [m.rstrip('.,!?') for m in re.findall(pattern, regex_text)]
    print(f'{name} ({len(matches)}):')
    for match in matches:
        print('  ', match)
    print()

Emails (2):
   nlpworkshop@gmail.com
   support@python.org

URLs (3):
   https://www.nlpworkshop.com
   www.python.org
   https://github.com/NLPWorkshop

Mobile numbers (2):
   +91-9876543210
   9123456789

Hashtags (3):
   #NLP
   #Python
   #MachineLearning

Mentions (2):
   @NLPWorkshop
   @PythonLearner



Two patterns need care to avoid stealing each other's matches. The mention pattern uses a
negative lookbehind so the `@gmail` inside an email address does not count as a mention, and
URL matches are stripped of trailing sentence punctuation, since `\S+` cannot know that the
final period of `www.python.org.` belongs to the sentence and not the address.